# 🚀 Simple LangGraph Workflow Tutorial

Is notebook mein hum LangGraph ka use karke ek basic stateful workflow build karenge:
- **State Schema** define karna TypedDict ke sath
- **Nodes (Functions)** create karna jo state ko modify karein
- **StateGraph** me nodes aur edges link karna (`START` -> `node_1` -> `node_2` -> `END`)
- Graph ko compile karke run (`invoke`) karna


### 📦 Module Imports
* **Purpose**: LangGraph aur typing utilities import karna.
* **Components**:
  - `StateGraph`: Workflow graph create karne ka core builder class.
  - `START`, `END`: LangGraph ke special entry point aur terminal nodes.
  - `TypedDict`: State structure ka type definition.


In [1]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


### 📋 State Schema Definition: `State`
* **Class Name**: `State(TypedDict)`
* **Purpose**: Workflow ke dauran carry hone wale data ka structure define karta hai.
* **Attributes**:
  - `graph_info: str`: Message string jo har node ke through pass aur update hoti hai.
* **Note**: Class name PascalCase (`State`) rakha gaya hai taaki function arguments (`state: State`) ke sath shadowing na ho.


In [2]:
class State(TypedDict):
    graph_info: str


### ⚙️ Node Functions: `node_1` & `node_2`
* **Functions**:
  - `node_1(state: State) -> dict`: Pehla processing step jo message append karta hai.
  - `node_2(state: State) -> dict`: Dusra processing step jo further processing append karta hai.
* **Parameters**:
  - `state` (`State`): Graph ki current state dictionary.
* **Returns**:
  - `dict`: Updated state keys jo graph state me merge ho jati hain.


In [3]:
def node_1(state: State) -> dict:
    """First node that transforms state."""
    current = state.get("graph_info", "")
    return {"graph_info": f"{current} -> Processed by Node 1"}

def node_2(state: State) -> dict:
    """Second node that transforms state further."""
    current = state.get("graph_info", "")
    return {"graph_info": f"{current} -> Processed by Node 2"}


### 🏗️ Graph Construction & Compilation
* **Purpose**: Nodes add karna, edges connect karna (`START -> node_1 -> node_2 -> END`), aur graph compile karna.
* **Methods**:
  - `builder.add_node(name, fn)`: Graph me function ko node ke roop me register karta hai.
  - `builder.add_edge(source, target)`: Nodes ke beech execution flow define karta hai.
  - `builder.compile()`: Graph structure validate karke executable runnable banata hai.


In [4]:
# Initialize graph builder with schema
builder = StateGraph(State)

# Add nodes
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)

# Connect edges
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

# Compile runnable graph
graph = builder.compile()


### 📊 Visualizing the Graph Flow
* **Purpose**: Graph structure ko visual flow diagram (ASCII text representation) me print karna.


In [5]:
try:
    print(graph.get_graph().draw_ascii())
except Exception as e:
    print("ASCII view not available:", e)


+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +--------+   
  | node_1 |   
  +--------+   
      *        
      *        
      *        
  +--------+   
  | node_2 |   
  +--------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


### ▶️ Graph Execution with `graph.invoke()`
* **Purpose**: Initial input state pass karke poore graph workflow ko run karna.
* **Parameters**:
  - `input` (`dict`): `{"graph_info": "Hello LangGraph"}`
* **Returns**:
  - `dict`: Final updated state dictionary.


In [6]:
initial_input = {"graph_info": "Hello LangGraph"}
final_state = graph.invoke(initial_input)

print("📥 Initial State:", initial_input)
print("📤 Final State Output:", final_state)


📥 Initial State: {'graph_info': 'Hello LangGraph'}
📤 Final State Output: {'graph_info': 'Hello LangGraph -> Processed by Node 1 -> Processed by Node 2'}
